# 04 — Model v1: full pipeline run

Runs `src/pipeline.py` end to end on the full data (on Kaggle via `notebooks/kaggle/push.py 04_v1_model`):

1. **Validation scenario** from train: 19% of S1 entities become "ghosts" (their S2/S3 records turn into
   look-alike distractors, matching the ~2.3 distractors per entity estimated for test), the rest split 80/20
   into train/validation folds. Aliases are learned from train-fold pairs only.
2. **Blocking** with combination keys → candidate pairs; recall reported on the validation scenario.
3. **Pair features** + **LightGBM**, early-stopped on the validation fold.
4. **One-owner assignment** + F0.5-optimal threshold on the validation fold (official macro F0.5, singletons included).
5. **Final model** on all computed rows → **test** candidates and matches written to `output/`.

In [ ]:
import sys, json, resource, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from config import ARTIFACT_DIR, DATA_DIR, OUTPUT_DIR, ON_KAGGLE
from pipeline import RunConfig, run
pd.set_option("display.width", 200)
print(f"data {DATA_DIR}\noutput {OUTPUT_DIR}\nartifacts {ARTIFACT_DIR}\non Kaggle: {ON_KAGGLE}")
cfg = RunConfig()
cfg

In [ ]:
report = run(cfg, OUTPUT_DIR, ARTIFACT_DIR)
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory {peak / (1e9 if platform.system() == 'Darwin' else 1e6):.1f} GB")

## Validation (official macro F0.5 on the validation fold)

In [ ]:
pd.Series(report["validation"]).to_frame("value")

In [ ]:
pd.DataFrame(report["blocking"]).T

In [ ]:
table = pd.read_csv(ARTIFACT_DIR / "threshold_table.tsv", sep="\t")
ax = table.plot(x="threshold", y=["f05", "precision", "recall"], figsize=(9, 4), marker=".")
ax.axvline(report["threshold"], color="grey", ls="--"); ax.set_title(f"validation F0.5 vs threshold (chosen {report['threshold']})")
plt.show()
table.round(4)

## Most important features (gain)

In [ ]:
pd.Series(report["top_features"]).to_frame("gain")

## Test output

In [ ]:
print(json.dumps(report["test"], indent=2))
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    p = OUTPUT_DIR / f
    print(f"{f}: {p.stat().st_size / 1e6:.1f} MB")
    display(pd.read_csv(p, sep="\t", nrows=5, keep_default_na=False))